In [10]:
import mujoco
import numpy as np

In [3]:
def print_model_info(model):
    """Print information about bodies and geoms in the model"""
    print("\n=== MODEL INFORMATION ===")
    print("Bodies:")
    for i in range(model.nbody):
        body_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i)
        if body_name:
            print(f"  {i}: {body_name}")
    
    print("\nGeoms:")
    for i in range(model.ngeom):
        geom_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, i)
        if geom_name:
            body_id = model.geom_bodyid[i]
            body_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, body_id)
            print(f"  {i}: {geom_name} (body: {body_name})")

In [7]:
# Load the model
xml_path = r"C:\wkspace\mj_ctrl\kuka_iiwa_14\scene_notarget.xml"  # Replace with your XML file path
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)

# Print model information for debugging
print_model_info(model)


=== MODEL INFORMATION ===
Bodies:
  0: world
  1: base
  2: link1
  3: link2
  4: link3
  5: link4
  6: link5
  7: link6
  8: link7
  9: attachment
  10: table
  11: simpleWoodTable

Geoms:
  0: floor (body: world)
  60: link7_visual (body: link7)
  61: attachment_visual (body: attachment)
  62: attachment_collision (body: attachment)
  68: board (body: simpleWoodTable)
  69: leg1 (body: simpleWoodTable)
  70: leg2 (body: simpleWoodTable)
  71: leg3 (body: simpleWoodTable)
  72: leg4 (body: simpleWoodTable)


In [11]:
def get_table_position(model, data):
    """Get the table's position in world coordinates"""
    try:
        # Try to find table body
        table_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "table")
        table_pos = data.xpos[table_body_id].copy()
        
        # Table surface is typically higher than body center
        table_height = table_pos[2] + 0.4  # Estimate table surface height
        table_surface_pos = np.array([table_pos[0], table_pos[1], table_height])
        
        return table_surface_pos
    except:
        # Fallback: use position from XML (table at 0.75, 0, 0 with ~0.4m height)
        # return np.array([0.75, 0.0, 0.4])
        return 0

mujoco.mj_forward(model, data)    
get_table_position(model, data)

array([0.75, 0.  , 0.4 ])

In [13]:
board_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "board")
data.geom_xpos[board_geom_id]

array([0.75 , 0.   , 0.425])

In [14]:
ee_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "attachment_collision")
data.geom_xpos[ee_geom_id]

array([0.   , 0.   , 1.306])

In [16]:
data.xpos[9]

array([0.   , 0.   , 1.306])

In [20]:
model.geom_bodyid

array([ 0,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  3,  3,  3,  3,  3,
        3,  3,  3,  3,  3,  3,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,
        4,  5,  5,  5,  5,  5,  5,  5,  5,  6,  6,  6,  6,  6,  6,  6,  6,
        6,  6,  6,  6,  7,  7,  7,  7,  7,  8,  9,  9, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11], dtype=int32)

In [22]:
model.geom_bodyid[70]

np.int32(11)

In [18]:
model.ngeom

73

In [ ]:
body1_id = 9
body2_id = 10
for geom_id in range(model.ngeom):
    geom_body_id = model.geom_bodyid[geom_id]
    if geom_body_id == body1_id:
        body1_geoms.append(geom_id)
    elif geom_body_id == body2_id:
        body2_geoms.append(geom_id)